In [ ]:
# Import Required Libraries
import numpy as np # numpy is used for numerical operations and mathematical calculations
import pandas as pd # Pandas is used for data loading, manipulation and analysis

# Model Selection & Preprocessing Utilities
from sklearn.model_selection import (
    train_test_split, # train_test_split is used to divide the dataset into training and testing sets
    GridSearchCV # GridSearchCV is used for hyperparameter tuning with cross-validation
)
from sklearn.compose import ColumnTransformer # ColumnTransformer allows different preprocessing for numerical and categorical columns
from sklearn.preprocessing import (
    OneHotEncoder, # OneHotEncoder converts categorical variables into numerical format
    StandardScaler # StandardScaler standardizes numerical features for better model performance
)
from sklearn.pipeline import Pipeline # Pipeline helps combine preprocessing and modeling steps in a clean workflow

# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score # MAE, MSE, and R2 Score are used to evaluate regression model performance

# Regression Models
from sklearn.linear_model import LinearRegression # Linear Regression is used as a baseline model
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor # Both are ensemble-based models capable of capturing non-linear relationships

### Dataset load and preview

In [ ]:
df = pd.read_csv("housing_dataset.csv") # Read the housing dataset from CSV file into a pandas DataFrame


df.head() #Display the first five rows of the dataset

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


### Feature & Target Separation

In [ ]:
# Separate the input features (X) and the target variable (y)
X = df.drop("price", axis=1) # X has all the input columns
y = df["price"] # y is the price we want to predict

### Column Type Identification

In [ ]:
num_cols = X.select_dtypes(include=['int64']).columns # Identify numerical columns (integer-based features)
cat_cols = X.select_dtypes(include=['object']).columns # Identify categorical columns (object/string-based features)

### Numerical Feature Pipeline

In [ ]:
# Pipeline for numerical columns
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()) # StandardScaler is used to normalize numerical features
    ]
)

# Pipeline for categorical columns
categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(  # OneHotEncoder converts categorical variables into numerical form
            drop='first' # drop='first' is used to avoid the dummy variable trap
        ))

    ]
)

# Combine both transformers
preprocessor = ColumnTransformer(
    transformers= [
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

### Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 # Split dataset into training (80%) and testing (20%) sets
)

### Model Initialization

In [ ]:
# Linear Regression: Simple baseline model to see linear relationship
reg_lr = LinearRegression()

# Random Forest Regressor: Ensemble tree model, good for non-linear patterns
reg_rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Gradient Boosting Regressor: Boosting model, often more accurate on tabular data
reg_gb = GradientBoostingRegressor(n_estimators=100, random_state=42)

### Model Dictionary

In [ ]:
# Store all models in a dictionary for iterative training and evaluation
model_to_train = {
    'Linear Regression' : reg_lr,
    'Random Forest' : reg_rf,
    'Gradient Boosting': reg_gb,
}

### Model Training & Evaluation Loop

In [ ]:
result = [] # List to store performance metrics of each model

# Loop through each model in our dictionary
for name, model in model_to_train.items():
  # Pipeline Creation: first preprocess then train model
  pipe = Pipeline(
      [
          ('preprocessor', preprocessor),
          ('model', model)
      ]
  )

  # Train the model on the training data
  pipe.fit(X_train, y_train)

  # Make predictions on the test set
  y_pred = pipe.predict(X_test)

  # Calculate performance metrics
  r2 = r2_score(y_test, y_pred)
  rmse = np.sqrt(mean_squared_error(y_test, y_pred))
  mae = mean_absolute_error(y_test, y_pred)

  # Store the results for comparison
  result.append({
      "Model": name,
      "R2 Score" :r2,
      "RMSE": rmse,
      "MAE" : mae
  })

# Convert results to a DataFrame and sort by best R2 score
result_df = pd.DataFrame(result).sort_values("R2 Score", ascending=False)

# Show the comparison of all models
print(result_df)

               Model  R2 Score          RMSE           MAE
2  Gradient Boosting  0.665965  1.299386e+06  9.597490e+05
0  Linear Regression  0.652924  1.324507e+06  9.700434e+05
1      Random Forest  0.612366  1.399758e+06  1.021151e+06


### Best Model Selection & Final Pipeline

In [ ]:
# Get the name of the best model based on R2 score
best_model_name = result_df.iloc[0]['Model']

# Retrieve the actual model object using the name
best_model_obj = model_to_train[best_model_name]

# Create a pipeline using the best model for hyperparameter tuning
final_pipe = Pipeline(
    steps=[
    ('preprocessor', preprocessor),
    ('model', best_model_obj)
])

### Hyperparameter Tuning with GridSearchCV

In [ ]:
# Define the grid of hyperparameters for Gradient Boosting
param_grid = {
    'model__n_estimators': [100, 200, 300], # n_estimators: number of trees
    'model__learning_rate': [0.01, 0.05, 0.1], # learning_rate: step size for boosting
    'model__max_depth': [2, 3, 4], # max_depth: max depth of each tree
    'model__subsample': [0.8, 1.0] # subsample: fraction of samples used for each tree
}

# Create GridSearchCV object
grid = GridSearchCV(
    final_pipe,
    param_grid,
    cv=5, # 5-fold cross-validation
    scoring="neg_mean_absolute_error", # pick parameters that minimize MAE
    n_jobs=-1 # use all CPU cores for faster training
)

# Fit GridSearchCV on training data to find the best parameters
grid.fit(X_train, y_train)

# Print the best combination of hyperparameters
print("Best Params:", grid.best_params_)

Best Params: {'model__learning_rate': 0.01, 'model__max_depth': 4, 'model__n_estimators': 300, 'model__subsample': 0.8}


### Best Model Extraction & Evaluation

In [ ]:
# Get the best model after GridSearchCV tuning
best_model = grid.best_estimator_

# Predict on test data using the optimized model
y_pred = best_model.predict(X_test)

# Calculate final evaluation metrics
mae = mean_absolute_error(y_test, y_pred)       # Average absolute error
mse = mean_squared_error(y_test, y_pred)        # Mean squared error
rmse = np.sqrt(mse)                             # Root mean squared error
r2 = r2_score(y_test, y_pred)                   # R-squared

# Performance Output
print("Gradient Boosting Results")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

Gradient Boosting Results
MAE: 1006522.8124862
MSE: 1884115272729.9023
RMSE: 1372630.785291479
R2: 0.6272452096027691
